In [1]:


# ================================================================
# Parkinson's Disease Detection using SVM
# Subject-wise 5-Fold Cross Validation
#
# Run this AFTER:
# df = pd.read_csv("data1.csv")
# ================================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

df=pd.read_csv('data1.csv')
# ================================================================
# 1. BASIC DATA ANALYSIS
# ================================================================

print("\n==============================")
print("DATASET ANALYSIS")
print("==============================")

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

print("\nNumber of recordings:", len(df))

print("Number of subjects:", df["subject"].nunique())

print("\nClass distribution:")
print(df["status"].value_counts())

print("\nMissing values (before NHR sentinel fix):")
print(df.isnull().sum())

print("\nRecordings per subject:")
print(df.groupby("subject").size().describe())


# ================================================================
# 2. CHECK THAT EACH SUBJECT HAS ONLY ONE STATUS
# ================================================================

subject_status = df.groupby("subject")["status"].nunique()

subjects_with_mixed_status = int(
    (subject_status > 1).sum()
)

print("\nSubjects having more than one status (should be 0):")
print(subjects_with_mixed_status)


if subjects_with_mixed_status > 0:

    print(
        "WARNING: some subjects have recordings under more than one "
        "class label. This can leak information across folds."
    )


# ================================================================
# 3. FIX KNOWN NHR SENTINEL VALUE
# ================================================================

# NHR = 999999 is treated as an invalid / missing value.
# The median imputer inside the pipeline will replace it.

df["NHR"] = df["NHR"].replace(
    999999,
    df['NH']
)


# ================================================================
# 4. SEPARATE FEATURES, TARGET AND SUBJECT GROUPS
# ================================================================

# Second code removes:
# name    -> recording identifier
# subject -> subject identifier
# status  -> target variable

X = df.drop(
    columns=[
    
        "subject",
        "status"
    ]
).copy()

y = df["status"]

groups = df["subject"]


print("\n==============================")
print("FEATURE INFORMATION")
print("==============================")

print("Number of features:", X.shape[1])

print("\nFeatures:")
print(list(X.columns))


#if X.shape[1] != 22:

   # raise ValueError(
       # f"Expected exactly 22 features, got {X.shape[1]}."
   # )


# ================================================================
# 5. CREATE SVM PIPELINE
# ================================================================

pipeline = Pipeline([

    # ------------------------------------------------------------
    # Median imputation
    # ------------------------------------------------------------
    # Missing values are replaced using the median calculated
    # ONLY from the training portion of each fold.
    # ------------------------------------------------------------

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),


    # ------------------------------------------------------------
    # Standardization
    # ------------------------------------------------------------
    # StandardScaler is also fitted ONLY using the training data.
    # ------------------------------------------------------------

    (
        "scaler",
        StandardScaler()
    ),


    # ------------------------------------------------------------
    # RBF SVM
    # ------------------------------------------------------------

    (
        "svm",
        SVC(
            kernel="rbf",
            C=10,
            gamma="scale",
            class_weight="balanced"
        )
    )

])


# ================================================================
# 6. CREATE 5-FOLD SUBJECT-WISE CROSS VALIDATION
# ================================================================

cv = StratifiedGroupKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)


# ================================================================
# 7. VARIABLES FOR STORING RESULTS
# ================================================================

fold_accuracies = []

fold_leakage_checks = []

all_y_true = []

all_y_pred = []


print("\n\n==============================")
print("5-FOLD SUBJECT-WISE SVM CROSS-VALIDATION")
print("==============================")


# ================================================================
# 8. TRAIN AND TEST THE MODEL
# ================================================================

for fold, (train_idx, test_idx) in enumerate(

        cv.split(
            X,
            y,
            groups=groups
        ),

        start=1
):


    # ------------------------------------------------------------
    # Training and testing data
    # ------------------------------------------------------------

    X_train = X.iloc[train_idx]

    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]

    y_test = y.iloc[test_idx]


    # ------------------------------------------------------------
    # Identify training and testing subjects
    # ------------------------------------------------------------

    train_subjects = set(
        groups.iloc[train_idx]
    )

    test_subjects = set(
        groups.iloc[test_idx]
    )


    # ------------------------------------------------------------
    # Check subject leakage
    # ------------------------------------------------------------

    common_subjects = (
        train_subjects.intersection(
            test_subjects
        )
    )


    print("\n------------------------------")

    print(f"Fold {fold}")

    print("------------------------------")

    print(
        "Training subjects:",
        len(train_subjects)
    )

    print(
        "Testing subjects :",
        len(test_subjects)
    )

    print(
        "Subjects appearing in both train and test:",
        common_subjects
    )


    # ------------------------------------------------------------
    # Stop if subject leakage occurs
    # ------------------------------------------------------------

    if common_subjects:

        raise RuntimeError(

            f"Subject leakage detected in fold {fold}: "
            f"{common_subjects}"

        )


    # ------------------------------------------------------------
    # TRAIN THE SVM
    # ------------------------------------------------------------

    pipeline.fit(
        X_train,
        y_train
    )


    # ------------------------------------------------------------
    # TEST / PREDICT
    # ------------------------------------------------------------

    y_pred = pipeline.predict(
        X_test
    )


    # ------------------------------------------------------------
    # CALCULATE ACCURACY
    # ------------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    fold_accuracies.append(
        accuracy
    )


    fold_leakage_checks.append(
        len(common_subjects) == 0
    )


    # ------------------------------------------------------------
    # Store predictions for overall evaluation
    # ------------------------------------------------------------

    all_y_true.extend(
        y_test.tolist()
    )

    all_y_pred.extend(
        y_pred.tolist()
    )


    print(
        "Fold accuracy:",
        round(accuracy, 4)
    )


# ================================================================
# 9. FINAL CROSS-VALIDATION RESULTS
# ================================================================

print("\n\n==============================")
print("FINAL CROSS-VALIDATION RESULT")
print("==============================")


for i, accuracy in enumerate(

        fold_accuracies,
        start=1
):

    print(

        f"Fold {i} accuracy: "
        f"{accuracy:.4f} "
        f"({accuracy * 100:.2f}%)"

    )


# ================================================================
# 10. MEAN ACCURACY AND STANDARD DEVIATION
# ================================================================

mean_accuracy = float(
    np.mean(
        fold_accuracies
    )
)


std_accuracy = float(
    np.std(
        fold_accuracies
    )
)


print(
    "\nMean accuracy:",
    round(mean_accuracy, 4)
)

print(
    "Mean accuracy (%):",
    round(mean_accuracy * 100, 2),
    "%"
)

print(
    "Standard deviation:",
    round(std_accuracy, 4)
)

print(
    "Standard deviation (%):",
    round(std_accuracy * 100, 2),
    "%"
)


# ================================================================
# 11. OVERALL CLASSIFICATION REPORT
# ================================================================

print("\n\n==============================")
print("CLASSIFICATION REPORT (pooled out-of-fold predictions)")
print("==============================")


report_text = classification_report(

    all_y_true,

    all_y_pred,

    target_names=[
        "Healthy",
        "Parkinson's"
    ],

    digits=4

)


print(
    report_text
)


# ================================================================
# 12. CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(

    all_y_true,

    all_y_pred

)


print("\n==============================")
print("CONFUSION MATRIX (pooled out-of-fold predictions)")
print("==============================")


print(cm)


print(
    "\nRows = Actual, Columns = Predicted"
)


print("\n              Predicted")

print(
    "              Healthy   Parkinson's"
)


print(

    f"Actual Healthy    "
    f"{cm[0, 0]:5d}      "
    f"{cm[0, 1]:5d}"

)


print(

    f"Actual PD         "
    f"{cm[1, 0]:5d}      "
    f"{cm[1, 1]:5d}"

)


# ================================================================
# 13. FINAL SUMMARY
# ================================================================

print("\n\n==============================")
print("SUMMARY")
print("==============================")


print(
    "Dataset recordings :",
    len(df)
)

print(
    "Subjects            :",
    df["subject"].nunique()
)

print(
    "Features            :",
    X.shape[1]
)

print(
    "CV folds            :",
    5
)

print(
    "SVM kernel          :",
    "rbf"
)

print(
    "C                   :",
    10
)

print(
    "Gamma               :",
    "scale"
)

print(
    "Scaler              :",
    "StandardScaler"
)

print(
    "Missing value method:",
    "Median imputation"
)

print(
    "Data splitting      :",
    "Subject-wise"
)

print(
    "Random state        :",
    42
)

print(
    "No subject leakage  :",
    all(fold_leakage_checks)
)


print(

    f"\nFinal CV accuracy: "
    f"{mean_accuracy * 100:.2f}% "
    f"+/- "
    f"{std_accuracy * 100:.2f}%"

)


DATASET ANALYSIS
Dataset shape: (668, 24)

First 5 rows:
  subject  MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  \
0  Anna B    -0.070983      1.000000     -0.148930        0.297350   
1  Anna B    -0.335138     -0.272580     -0.172900        0.093426   
2  Anna B    -0.197711     -0.272580     -0.040123       -0.959830   
3  Anna B    -0.270211     -0.367916      0.049314       -0.979620   
4  Anna B    -0.180395     -0.264946     -0.242367       -0.951661   

   MDVP:Jitter(Abs)  MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  ...  \
0         -0.151314  0.102446  0.682908    0.102446     -0.297704  ...   
1         -0.155670 -0.070865  0.426115   -0.070865     -0.168589  ...   
2         -0.972630 -0.958083 -0.945009   -0.958083     -0.877982  ...   
3         -0.984241 -0.973908 -0.971633   -0.973908     -0.905835  ...   
4         -0.967728 -0.951115 -0.936007   -0.951115     -0.873578  ...   

   Shimmer:DDA       NHR       HNR      RPDE       DFA   spread1   spread2  